# Color Recognition and Mixing System Demo

This comprehensive notebook demonstrates the complete color recognition and mixing pipeline, including:

1. **Project Structure Setup** - Initialize environment and import libraries
2. **Preprocessing & Color Calibration** - Camera calibration using color checker
3. **Color Space Conversion** - RGB to CIE Lab transformation
4. **Color Recognition with SVM** - Machine learning-based color classification
5. **Kubelka-Munk Color Mixing Model** - Physical color mixing calculations
6. **Color Error Calculation (ΔE)** - Accuracy evaluation using CIEDE2000
7. **Multi-objective Optimization** - Optimize mixing formulas for cost and quality
8. **Pipeline Integration** - Complete end-to-end system demonstration

## System Overview

The system combines computer vision, machine learning, and physical color models to automatically recognize colors and generate optimal paint mixing formulas for industrial applications.

## 1. Project Structure Setup

First, let's set up the project environment and import all necessary libraries.

In [ ]:
# Import system libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('../src')

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Computer vision and image processing
import cv2
from PIL import Image

# Machine learning
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Color science
import colorspacious

# Our custom modules
from utils import (
    ColorSpaceConverter, 
    ImageProcessor, 
    ColorDifferenceCalculator,
    Visualizer,
    DataProcessor
)
from preprocessing import CameraCalibrator, ColorChecker, ImagePreprocessor
from color_recognition import SVMColorClassifier, ColorFeatureExtractor
from mixing_formula import KubelkaMunkModel, MixingOptimizer, create_standard_pigments
from optimization import MultiObjectiveOptimizer, AdaptiveOptimizer

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

In [ ]:
# Create project directories if they don't exist
project_dirs = [
    '../data/raw',
    '../data/processed', 
    '../data/color_checker',
    '../models/color_detection',
    '../models/color_mixing',
    '../results'
]

for dir_path in project_dirs:
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    
print("📁 Project directory structure created:")
for dir_path in project_dirs:
    print(f"   {dir_path}")

# Set random seeds for reproducibility
np.random.seed(42)

print("\n🎯 Environment setup complete!")

## 2. Preprocessing and Color Calibration

Color calibration is crucial for accurate color reproduction. We'll use a color checker pattern to calibrate the camera and correct for lighting conditions.

In [ ]:
# Create a synthetic color checker for demonstration
def create_synthetic_color_checker():
    """Create a synthetic color checker image for demo purposes"""
    # Standard Macbeth ColorChecker colors (simplified)
    checker_colors = [
        [115, 82, 68],   # Dark skin
        [194, 150, 130], # Light skin
        [98, 122, 157],  # Blue sky
        [87, 108, 67],   # Foliage
        [133, 128, 177], # Blue flower
        [103, 189, 170], # Bluish green
        [214, 126, 44],  # Orange
        [80, 91, 166],   # Purplish blue
        [193, 90, 99],   # Moderate red
        [94, 60, 108],   # Purple
        [157, 188, 64],  # Yellow green
        [224, 163, 46],  # Orange yellow
        [56, 61, 150],   # Blue
        [70, 148, 73],   # Green
        [175, 54, 60],   # Red
        [231, 199, 31],  # Yellow
        [187, 86, 149],  # Magenta
        [8, 133, 161],   # Cyan
        [243, 243, 242], # White
        [200, 200, 200], # Neutral 8
        [160, 160, 160], # Neutral 6.5
        [122, 122, 121], # Neutral 5
        [85, 85, 85],    # Neutral 3.5
        [52, 52, 52]     # Black
    ]
    
    # Create 6x4 grid
    patch_size = 50
    grid_h, grid_w = 4, 6
    
    checker_image = np.zeros((grid_h * patch_size, grid_w * patch_size, 3), dtype=np.uint8)
    
    for i, color in enumerate(checker_colors):
        row = i // grid_w
        col = i % grid_w
        
        y1, y2 = row * patch_size, (row + 1) * patch_size
        x1, x2 = col * patch_size, (col + 1) * patch_size
        
        checker_image[y1:y2, x1:x2] = color
    
    return checker_image

# Create and display synthetic color checker
synthetic_checker = create_synthetic_color_checker()

plt.figure(figsize=(12, 8))
plt.imshow(synthetic_checker)
plt.title('Synthetic Color Checker for Calibration')
plt.axis('off')
plt.show()

print("🎨 Synthetic color checker created for demonstration")

In [ ]:
# Initialize camera calibrator
calibrator = CameraCalibrator(method='neural_network')

# Simulate camera distortion for realistic demo
def add_camera_distortion(image, brightness_shift=0.1, color_shift=0.05):
    """Add realistic camera distortion to simulate uncalibrated conditions"""
    distorted = image.astype(np.float32) / 255.0
    
    # Add brightness variation
    distorted = distorted * (1 + brightness_shift * np.random.randn())
    
    # Add color channel shifts
    for c in range(3):
        distorted[:, :, c] *= (1 + color_shift * np.random.randn())
    
    distorted = np.clip(distorted * 255, 0, 255).astype(np.uint8)
    return distorted

# Create distorted version for calibration demo
distorted_checker = add_camera_distortion(synthetic_checker)

# Display original vs distorted
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].imshow(synthetic_checker)
axes[0].set_title('Original Color Checker')
axes[0].axis('off')

axes[1].imshow(distorted_checker)
axes[1].set_title('Distorted (Uncalibrated Camera)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Perform calibration
print("🔧 Performing camera calibration...")
success = calibrator.calibrate(distorted_checker)

if success:
    # Apply calibration to the distorted image
    calibrated_checker = calibrator.apply_calibration(distorted_checker)
    
    # Display calibration results
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(synthetic_checker)
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    axes[1].imshow(distorted_checker)
    axes[1].set_title('Uncalibrated')
    axes[1].axis('off')
    
    axes[2].imshow(calibrated_checker)
    axes[2].set_title('Calibrated')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Camera calibration completed successfully!")
else:
    print("❌ Camera calibration failed")

## 3. Color Space Conversion

Converting from RGB to CIE Lab color space allows us to separate perceptual components and work with perceptually uniform color differences.

In [ ]:
# Demonstrate color space conversion
def demonstrate_color_spaces(image):
    """Demonstrate different color space representations"""
    
    # Extract a sample color patch
    sample_rgb = np.mean(image[50:100, 50:100], axis=(0, 1))
    
    # Convert to different color spaces
    sample_lab = ColorSpaceConverter.rgb_to_lab(sample_rgb)
    sample_hsv = ColorSpaceConverter.rgb_to_hsv(sample_rgb)
    
    print("🎨 Color Space Conversion Example:")
    print(f"RGB: [{sample_rgb[0]:.1f}, {sample_rgb[1]:.1f}, {sample_rgb[2]:.1f}]")
    print(f"Lab: [L*={sample_lab[0]:.1f}, a*={sample_lab[1]:.1f}, b*={sample_lab[2]:.1f}]")
    print(f"HSV: [H={sample_hsv[0]:.1f}, S={sample_hsv[1]:.1f}, V={sample_hsv[2]:.1f}]")
    
    return sample_rgb, sample_lab, sample_hsv

# Convert entire image to Lab
def visualize_lab_channels(image):
    """Visualize Lab color channels"""
    
    # Convert entire image to Lab
    lab_image = np.zeros_like(image, dtype=np.float32)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            lab_image[i, j] = ColorSpaceConverter.rgb_to_lab(image[i, j])
    
    # Split channels
    L_channel = lab_image[:, :, 0]
    a_channel = lab_image[:, :, 1]
    b_channel = lab_image[:, :, 2]
    
    # Visualize channels
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Original image
    axes[0, 0].imshow(image)
    axes[0, 0].set_title('Original RGB Image')
    axes[0, 0].axis('off')
    
    # L* channel (Lightness)
    axes[0, 1].imshow(L_channel, cmap='gray')
    axes[0, 1].set_title('L* Channel (Lightness)')
    axes[0, 1].axis('off')
    
    # a* channel (Green-Red)
    axes[1, 0].imshow(a_channel, cmap='RdYlGn_r')
    axes[1, 0].set_title('a* Channel (Green ← → Red)')
    axes[1, 0].axis('off')
    
    # b* channel (Blue-Yellow)
    axes[1, 1].imshow(b_channel, cmap='coolwarm')
    axes[1, 1].set_title('b* Channel (Blue ← → Yellow)')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return lab_image

# Demonstrate with calibrated checker
sample_rgb, sample_lab, sample_hsv = demonstrate_color_spaces(calibrated_checker)
lab_checker = visualize_lab_channels(calibrated_checker)

print("\\n📊 Lab color space provides perceptually uniform color differences")

## 4. Color Recognition with SVM

We'll train an SVM classifier with advanced feature extraction to recognize and classify colors accurately.

In [ ]:
# Create synthetic training data for color recognition
def create_color_training_data():
    """Create synthetic color samples for training"""
    
    color_categories = {
        'Red': [(220, 20, 60), (255, 99, 71), (178, 34, 34), (139, 0, 0)],
        'Green': [(0, 128, 0), (34, 139, 34), (50, 205, 50), (0, 100, 0)],
        'Blue': [(0, 0, 255), (30, 144, 255), (70, 130, 180), (0, 0, 139)],
        'Yellow': [(255, 255, 0), (255, 215, 0), (218, 165, 32), (184, 134, 11)],
        'Orange': [(255, 165, 0), (255, 140, 0), (255, 69, 0), (255, 99, 71)],
        'Purple': [(128, 0, 128), (148, 0, 211), (138, 43, 226), (75, 0, 130)],
        'Pink': [(255, 192, 203), (255, 20, 147), (219, 112, 147), (199, 21, 133)],
        'Brown': [(139, 69, 19), (160, 82, 45), (210, 180, 140), (205, 133, 63)],
        'Gray': [(128, 128, 128), (169, 169, 169), (105, 105, 105), (64, 64, 64)],
        'White': [(255, 255, 255), (248, 248, 255), (245, 245, 245), (220, 220, 220)]
    }
    
    training_images = []
    training_labels = []
    
    # Generate training patches
    for label, colors in color_categories.items():
        for base_color in colors:
            # Create variations with noise
            for _ in range(10):  # 10 variations per base color
                # Add random noise
                noise = np.random.normal(0, 15, 3)
                noisy_color = np.clip(np.array(base_color) + noise, 0, 255).astype(np.uint8)
                
                # Create 32x32 patch with slight variations
                patch = np.full((32, 32, 3), noisy_color, dtype=np.uint8)
                
                # Add some texture variation
                texture_noise = np.random.normal(0, 5, (32, 32, 3))
                patch = np.clip(patch.astype(np.float32) + texture_noise, 0, 255).astype(np.uint8)
                
                training_images.append(patch)
                training_labels.append(label)
    
    return training_images, training_labels

# Create training data
print("🎯 Creating synthetic training data...")
train_images, train_labels = create_color_training_data()

print(f"Created {len(train_images)} training samples across {len(set(train_labels))} color categories")

# Visualize some training samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

unique_labels = list(set(train_labels))
for i, label in enumerate(unique_labels):
    # Find first sample of this label
    sample_idx = train_labels.index(label)
    axes[i].imshow(train_images[sample_idx])
    axes[i].set_title(label)
    axes[i].axis('off')

plt.suptitle('Training Data Samples')
plt.tight_layout()
plt.show()

In [ ]:
# Train SVM color classifier
print("🤖 Training SVM Color Classifier...")

# Initialize classifier with combined features
svm_classifier = SVMColorClassifier(feature_type='combined')

# Train the model
training_results = svm_classifier.train(train_images, train_labels, test_size=0.2)

print("\\n📊 Training Results:")
print(f"Best Parameters: {training_results['best_params']}")
print(f"Training Accuracy: {training_results['train_accuracy']:.3f}")
print(f"Test Accuracy: {training_results['test_accuracy']:.3f}")
print("\\n📈 Classification Report:")
print(training_results['classification_report'])

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
cm = training_results['confusion_matrix']
unique_labels = sorted(list(set(train_labels)))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=unique_labels, yticklabels=unique_labels)
plt.title('Confusion Matrix - Color Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("✅ SVM training completed successfully!")

## 5. Kubelka-Munk Color Mixing Model

The Kubelka-Munk theory models the physical interaction of light with pigments, allowing us to predict color mixing results based on absorption and scattering properties.

In [ ]:
# Initialize Kubelka-Munk model with standard pigments
print("🎨 Setting up Kubelka-Munk Color Mixing Model...")

# Create standard pigments
pigments = create_standard_pigments()

# Initialize KM model
km_model = KubelkaMunkModel()

# Add pigments to model
for pigment in pigments:
    km_model.add_pigment(pigment)

print(f"Added {len(pigments)} standard pigments to the model:")
for pigment in pigments:
    print(f"  • {pigment.name} (Cost: ${pigment.cost_per_unit:.1f}/unit)")

# Demonstrate color prediction
print("\\n🧪 Testing color mixing predictions...")

# Test mixture: 30% Titanium White + 20% Chrome Yellow + 10% Cadmium Red
test_concentrations = {
    'Titanium_White': 0.3,
    'Chrome_Yellow': 0.2,
    'Cadmium_Red': 0.1,
    'Carbon_Black': 0.0,
    'Ultramarine_Blue': 0.0
}

# Predict resulting color
predicted_lab = km_model.predict_color(test_concentrations)
predicted_rgb = ColorSpaceConverter.lab_to_rgb(predicted_lab)

print(f"\\nTest Mixture Results:")
print(f"Concentrations: {test_concentrations}")
print(f"Predicted Lab: [L*={predicted_lab[0]:.1f}, a*={predicted_lab[1]:.1f}, b*={predicted_lab[2]:.1f}]")
print(f"Predicted RGB: [{predicted_rgb[0]}, {predicted_rgb[1]}, {predicted_rgb[2]}]")

# Visualize the predicted color
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Show color swatch
color_swatch = np.full((100, 100, 3), predicted_rgb, dtype=np.uint8)
axes[0].imshow(color_swatch)
axes[0].set_title('Predicted Mixed Color')
axes[0].axis('off')

# Show mixture composition
active_pigments = {k: v for k, v in test_concentrations.items() if v > 0}
axes[1].pie(active_pigments.values(), labels=active_pigments.keys(), autopct='%1.1f%%')
axes[1].set_title('Pigment Composition')

plt.tight_layout()
plt.show()

print("✅ Kubelka-Munk model setup complete!")

## 6. Color Error Calculation (ΔE)

The ΔE metric quantifies perceptual color differences. CIEDE2000 is the most advanced formula for color difference calculation.

In [ ]:
# Demonstrate color difference calculations
def demonstrate_delta_e():
    """Demonstrate ΔE calculations with different color pairs"""
    
    # Define test color pairs (Lab values)
    color_pairs = [
        ([50, 0, 0], [52, 0, 0], "Very close colors"),
        ([50, 0, 0], [55, 0, 0], "Small difference"),
        ([50, 0, 0], [60, 10, 10], "Moderate difference"),
        ([50, 0, 0], [70, 20, -20], "Large difference"),
        ([50, 20, 30], [30, -10, -15], "Very different colors")
    ]
    
    print("📏 Color Difference (ΔE) Demonstration:")
    print("=" * 60)
    
    results = []
    
    for lab1, lab2, description in color_pairs:
        lab1_arr = np.array(lab1)
        lab2_arr = np.array(lab2)
        
        # Calculate different ΔE formulas
        delta_e_76 = ColorDifferenceCalculator.delta_e_cie76(lab1_arr, lab2_arr)
        delta_e_2000 = ColorDifferenceCalculator.delta_e_ciede2000(lab1_arr, lab2_arr)
        
        # Convert to RGB for visualization
        rgb1 = ColorSpaceConverter.lab_to_rgb(lab1_arr)
        rgb2 = ColorSpaceConverter.lab_to_rgb(lab2_arr)
        
        results.append({
            'description': description,
            'lab1': lab1,
            'lab2': lab2,
            'rgb1': rgb1,
            'rgb2': rgb2,
            'delta_e_76': delta_e_76,
            'delta_e_2000': delta_e_2000
        })
        
        print(f"{description:20} | ΔE76: {delta_e_76:5.2f} | ΔE2000: {delta_e_2000:5.2f}")
    
    return results

# Calculate color differences
delta_e_results = demonstrate_delta_e()

# Visualize color pairs and their differences
fig, axes = plt.subplots(len(delta_e_results), 3, figsize=(12, 15))

for i, result in enumerate(delta_e_results):
    # Color 1
    color1_swatch = np.full((50, 50, 3), result['rgb1'], dtype=np.uint8)
    axes[i, 0].imshow(color1_swatch)
    axes[i, 0].set_title(f"Color 1\\nLab: {result['lab1']}")
    axes[i, 0].axis('off')
    
    # Color 2
    color2_swatch = np.full((50, 50, 3), result['rgb2'], dtype=np.uint8)
    axes[i, 1].imshow(color2_swatch)
    axes[i, 1].set_title(f"Color 2\\nLab: {result['lab2']}")
    axes[i, 1].axis('off')
    
    # Difference visualization
    axes[i, 2].bar(['ΔE76', 'ΔE2000'], 
                   [result['delta_e_76'], result['delta_e_2000']], 
                   color=['lightblue', 'lightcoral'])
    axes[i, 2].set_title(f"{result['description']}")
    axes[i, 2].set_ylabel('ΔE Value')
    
    # Add quality interpretation
    delta_e = result['delta_e_2000']
    if delta_e < 1:
        quality = "Excellent (< 1)"
    elif delta_e < 2:
        quality = "Good (< 2)"
    elif delta_e < 4:
        quality = "Acceptable (< 4)"
    else:
        quality = "Poor (≥ 4)"
    
    axes[i, 2].text(0.5, max(result['delta_e_76'], result['delta_e_2000']) * 0.8, 
                    quality, ha='center', fontsize=8)

plt.suptitle('Color Difference (ΔE) Analysis', fontsize=16)
plt.tight_layout()
plt.show()

print("\\n📊 ΔE Quality Standards:")
print("  • ΔE < 1.0:  Excellent match (not perceptible)")
print("  • ΔE < 2.0:  Good match (barely perceptible)")  
print("  • ΔE < 4.0:  Acceptable match (small difference)")
print("  • ΔE ≥ 4.0:  Poor match (obvious difference)")

## 7. Multi-objective Optimization

We'll optimize mixing formulas considering multiple objectives: color accuracy (ΔE), cost minimization, and formula simplicity.

In [ ]:
# Initialize optimizer
print("🎯 Setting up Multi-objective Optimization...")

optimizer = MixingOptimizer(km_model)

# Define target colors for optimization
target_colors = [
    np.array([65, 15, 25]),   # Light red
    np.array([80, -10, 60]),  # Yellow-green
    np.array([45, 0, -30]),   # Blue
    np.array([90, 0, 0]),     # Light gray
    np.array([35, 25, -15])   # Dark purple
]

color_names = ['Light Red', 'Yellow-Green', 'Blue', 'Light Gray', 'Dark Purple']

print(f"\\n🎨 Optimizing formulas for {len(target_colors)} target colors...")

# Optimize each color
optimization_results = []

for i, (target_lab, name) in enumerate(zip(target_colors, color_names)):
    print(f"\\nOptimizing {name} (Color {i+1}/{len(target_colors)})...")
    
    # Run optimization with different methods
    result_de = optimizer.optimize_formula(
        target_lab, 
        method='differential_evolution',
        cost_weight=0.05,
        complexity_weight=0.02,
        max_iterations=500
    )
    
    result_ls = optimizer.optimize_formula(
        target_lab,
        method='least_squares', 
        max_iterations=300
    )
    
    # Store results
    optimization_results.append({
        'name': name,
        'target_lab': target_lab,
        'de_result': result_de,
        'ls_result': result_ls
    })
    
    print(f"  Differential Evolution: ΔE={result_de['delta_e']:.2f}, Cost=${result_de['total_cost']:.2f}")
    print(f"  Least Squares:          ΔE={result_ls['delta_e']:.2f}, Cost=${result_ls['total_cost']:.2f}")

print("\\n✅ Optimization completed for all target colors!")

In [ ]:
# Visualize optimization results
def visualize_optimization_comparison(results):
    """Visualize comparison between optimization methods"""
    
    n_colors = len(results)
    fig, axes = plt.subplots(n_colors, 4, figsize=(16, 4*n_colors))
    
    if n_colors == 1:
        axes = axes.reshape(1, -1)
    
    for i, result in enumerate(results):
        target_lab = result['target_lab']
        de_result = result['de_result']
        ls_result = result['ls_result']
        
        # Target color
        target_rgb = ColorSpaceConverter.lab_to_rgb(target_lab)
        target_swatch = np.full((100, 100, 3), target_rgb, dtype=np.uint8)
        axes[i, 0].imshow(target_swatch)
        axes[i, 0].set_title(f'{result["name"]}\\nTarget')
        axes[i, 0].axis('off')
        
        # DE result
        de_rgb = ColorSpaceConverter.lab_to_rgb(de_result['predicted_lab'])
        de_swatch = np.full((100, 100, 3), de_rgb, dtype=np.uint8)
        axes[i, 1].imshow(de_swatch)
        axes[i, 1].set_title(f'Diff. Evolution\\nΔE: {de_result["delta_e"]:.2f}')
        axes[i, 1].axis('off')
        
        # LS result  
        ls_rgb = ColorSpaceConverter.lab_to_rgb(ls_result['predicted_lab'])
        ls_swatch = np.full((100, 100, 3), ls_rgb, dtype=np.uint8)
        axes[i, 2].imshow(ls_swatch)
        axes[i, 2].set_title(f'Least Squares\\nΔE: {ls_result["delta_e"]:.2f}')
        axes[i, 2].axis('off')
        
        # Comparison metrics
        methods = ['DE', 'LS']
        delta_es = [de_result['delta_e'], ls_result['delta_e']]
        costs = [de_result['total_cost'], ls_result['total_cost']]
        
        ax_comp = axes[i, 3]
        x = np.arange(len(methods))
        width = 0.35
        
        ax_comp2 = ax_comp.twinx()
        
        bars1 = ax_comp.bar(x - width/2, delta_es, width, label='ΔE', color='lightblue')
        bars2 = ax_comp2.bar(x + width/2, costs, width, label='Cost', color='lightcoral')
        
        ax_comp.set_xlabel('Method')
        ax_comp.set_ylabel('ΔE', color='blue')
        ax_comp2.set_ylabel('Cost ($)', color='red')
        ax_comp.set_title(f'{result["name"]} Comparison')
        ax_comp.set_xticks(x)
        ax_comp.set_xticklabels(methods)
        
        # Add value labels on bars
        for j, (bar1, bar2) in enumerate(zip(bars1, bars2)):
            height1 = bar1.get_height()
            height2 = bar2.get_height()
            ax_comp.text(bar1.get_x() + bar1.get_width()/2., height1 + 0.05,
                        f'{height1:.2f}', ha='center', va='bottom')
            ax_comp2.text(bar2.get_x() + bar2.get_width()/2., height2 + 0.1,
                         f'${height2:.2f}', ha='center', va='bottom')
    
    plt.suptitle('Optimization Methods Comparison', fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualize results
visualize_optimization_comparison(optimization_results)

# Create summary table
summary_data = []
for result in optimization_results:
    de_res = result['de_result']
    ls_res = result['ls_result']
    
    summary_data.append({
        'Color': result['name'],
        'DE_ΔE': f"{de_res['delta_e']:.2f}",
        'DE_Cost': f"${de_res['total_cost']:.2f}",
        'DE_Pigments': de_res['num_pigments_used'],
        'LS_ΔE': f"{ls_res['delta_e']:.2f}",
        'LS_Cost': f"${ls_res['total_cost']:.2f}",
        'LS_Pigments': ls_res['num_pigments_used']
    })

summary_df = pd.DataFrame(summary_data)
print("\\n📊 Optimization Summary:")
print(summary_df.to_string(index=False))

## 8. Pipeline Integration

Finally, let's integrate all components into a complete pipeline that processes images from start to finish.

In [ ]:
# Create complete pipeline
class ColorRecognitionPipeline:
    """Complete color recognition and mixing pipeline"""
    
    def __init__(self):
        self.calibrator = calibrator
        self.preprocessor = ImagePreprocessor(calibrator)
        self.color_detector = svm_classifier
        self.km_model = km_model
        self.optimizer = optimizer
    
    def process_image(self, image, region=None):
        """Process image through complete pipeline"""
        results = {}
        
        # 1. Preprocessing
        processed = self.preprocessor.preprocess_image(
            image, 
            apply_calibration=True,
            correct_lighting=True,
            lighting_method='gray_world'
        )
        results['processed_image'] = processed
        
        # 2. Color detection
        if region:
            detection_region = region
        else:
            detection_region = None
            
        color_label, confidence = self.color_detector.predict(processed, detection_region)
        results['color_detection'] = {
            'label': color_label,
            'confidence': confidence
        }
        
        # 3. Extract target color
        if region:
            x, y, w, h = region
            target_rgb = np.mean(processed[y:y+h, x:x+w], axis=(0, 1))
        else:
            target_rgb = np.mean(processed, axis=(0, 1))
        
        target_lab = ColorSpaceConverter.rgb_to_lab(target_rgb)
        results['target_color'] = {
            'rgb': target_rgb,
            'lab': target_lab
        }
        
        # 4. Optimize mixing formula
        mixing_result = self.optimizer.optimize_formula(
            target_lab,
            method='differential_evolution',
            cost_weight=0.1,
            complexity_weight=0.05
        )
        results['mixing_formula'] = mixing_result
        
        return results

# Initialize pipeline
pipeline = ColorRecognitionPipeline()
print("🚀 Complete Color Recognition Pipeline initialized!")

# Create test images
def create_test_image(color_rgb, size=(200, 200)):
    """Create test image with uniform color"""
    return np.full((*size, 3), color_rgb, dtype=np.uint8)

# Test colors
test_colors = [
    ([255, 100, 100], "Light Red"),
    ([100, 255, 100], "Light Green"), 
    ([100, 100, 255], "Light Blue"),
    ([255, 255, 100], "Yellow"),
    ([255, 150, 50], "Orange")
]

print(f"\\n🧪 Testing pipeline with {len(test_colors)} test images...")

In [ ]:
# Process test images through pipeline
pipeline_results = []

for i, (color_rgb, name) in enumerate(test_colors):
    print(f"\\nProcessing {name} (Image {i+1}/{len(test_colors)})...")
    
    # Create test image
    test_image = create_test_image(color_rgb)
    
    # Process through pipeline
    result = pipeline.process_image(test_image)
    result['name'] = name
    result['original_rgb'] = color_rgb
    pipeline_results.append(result)
    
    # Print summary
    mixing = result['mixing_formula']
    detection = result['color_detection']
    
    print(f"  Detected: {detection['label']} (confidence: {detection['confidence']:.2f})")
    print(f"  ΔE: {mixing['delta_e']:.2f}")
    print(f"  Cost: ${mixing['total_cost']:.2f}")
    print(f"  Pigments used: {mixing['num_pigments_used']}")

print("\\n✅ Pipeline testing completed!")

# Visualize pipeline results
fig, axes = plt.subplots(len(test_colors), 4, figsize=(16, 4*len(test_colors)))

if len(test_colors) == 1:
    axes = axes.reshape(1, -1)

for i, result in enumerate(pipeline_results):
    # Original image
    orig_swatch = np.full((100, 100, 3), result['original_rgb'], dtype=np.uint8)
    axes[i, 0].imshow(orig_swatch)
    axes[i, 0].set_title(f'{result["name"]}\\nOriginal')
    axes[i, 0].axis('off')
    
    # Processed image
    processed = result['processed_image']
    proc_color = np.mean(processed, axis=(0, 1)).astype(np.uint8)
    proc_swatch = np.full((100, 100, 3), proc_color, dtype=np.uint8)
    axes[i, 1].imshow(proc_swatch)
    axes[i, 1].set_title(f'Processed\\n{result["color_detection"]["label"]}')
    axes[i, 1].axis('off')
    
    # Predicted mix
    mixing = result['mixing_formula']
    pred_rgb = ColorSpaceConverter.lab_to_rgb(mixing['predicted_lab'])
    pred_swatch = np.full((100, 100, 3), pred_rgb, dtype=np.uint8)
    axes[i, 2].imshow(pred_swatch)
    axes[i, 2].set_title(f'Predicted Mix\\nΔE: {mixing["delta_e"]:.2f}')
    axes[i, 2].axis('off')
    
    # Formula composition
    concentrations = mixing['concentrations']
    nonzero_conc = {k: v for k, v in concentrations.items() if v > 0.01}
    
    if nonzero_conc:
        axes[i, 3].pie(nonzero_conc.values(), labels=nonzero_conc.keys(), 
                      autopct='%1.1f%%', startangle=90)
        axes[i, 3].set_title(f'Formula\\nCost: ${mixing["total_cost"]:.2f}')
    else:
        axes[i, 3].text(0.5, 0.5, 'No mix needed', ha='center', va='center')
        axes[i, 3].set_title('Formula')

plt.suptitle('Complete Pipeline Results', fontsize=16)
plt.tight_layout()
plt.show()

# Create final summary
print("\\n" + "="*80)
print("🎯 FINAL SYSTEM PERFORMANCE SUMMARY")
print("="*80)

avg_delta_e = np.mean([r['mixing_formula']['delta_e'] for r in pipeline_results])
avg_cost = np.mean([r['mixing_formula']['total_cost'] for r in pipeline_results])
avg_pigments = np.mean([r['mixing_formula']['num_pigments_used'] for r in pipeline_results])

print(f"Average ΔE:              {avg_delta_e:.2f}")
print(f"Average Cost:            ${avg_cost:.2f}")
print(f"Average Pigments Used:   {avg_pigments:.1f}")

successful = sum(1 for r in pipeline_results if r['mixing_formula']['delta_e'] <= 2.0)
success_rate = successful / len(pipeline_results) * 100

print(f"Success Rate (ΔE ≤ 2.0): {success_rate:.1f}%")
print("\\n🏆 System successfully demonstrates:")
print("  ✓ Camera calibration and preprocessing")
print("  ✓ Color space conversion (RGB → Lab)")
print("  ✓ SVM-based color recognition")
print("  ✓ Kubelka-Munk physical modeling")
print("  ✓ Multi-objective optimization")
print("  ✓ Complete pipeline integration")
print("\\n🎨 Ready for industrial paint mixing applications!")